In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
import pandas as pd
import os
import shutil
from numba import njit
from google.colab import files

# ==========================================
# 1. シミュレーション・パラメータ設定
# ==========================================
N_LIST = [1000, 10000, 100000] # スケール不変性の確認 (Colabのメモリと実行時間を考慮し10万まで)
BURN_IN_STEPS = 100  # Phase 0: 自己組織化 (t=0~99)
ATTACK_STEPS = 100   # Phase 1a: 偽ペイオフ注入 (t=100~199)
HYSTERESIS_STEPS = 100 # Phase 1b: ペイオフ停止・ヒステリシス確認 (t=200~299)
TOTAL_STEPS = BURN_IN_STEPS + ATTACK_STEPS + HYSTERESIS_STEPS
DT = 0.1

B_PAYLOAD = 0.9      # 攻撃ペイオフの強度

OUTPUT_DIR = "simulation_results_hypothesis1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 2. Numbaによる高速SDE積分エンジン
# ==========================================
@njit
def sde_update(X, D, degrees, b_val, N, dt=0.1, sigma=0.05, gamma=6.0, k_break=0.6, k_restore=0.1):
    # スケールフリー・トポロジーにおける局所平均場 P の近似
    total_degree = np.sum(degrees)
    mean_field = np.sum(X * degrees) / total_degree if total_degree > 0 else 0.0

    # 確率的認知ノイズ
    noise_X = np.random.normal(0, 1, N) * np.sqrt(dt)

    # dX: コンフォーミティ(同調圧力)の更新
    dX = (mean_field + b_val - X) * dt + sigma * noise_X
    X_new = np.minimum(np.maximum(X + dX, 0.0), 1.0)

    # dD: トポロジカル欠陥(β1)の更新
    repair_capacity = np.exp(-gamma * D)
    dD = (k_break * X_new * (1.0 - D) - k_restore * (1.0 - X_new) * D * repair_capacity) * dt
    D_new = np.minimum(np.maximum(D + dD, 0.0), 1.0)

    return X_new, D_new

# ==========================================
# 3. メインシミュレーションループ
# ==========================================
ks_results = []
defect_results = []

for N in N_LIST:
    print(f"--- Simulating N = {N} ---")

    # 次数分布の生成 (べき乗則近似)
    degrees = np.random.pareto(2.5, N) + 1.0
    hub_idx = np.argmax(degrees)
    random_idx = np.random.randint(0, N)

    # Hub Origin と Random Origin の2シナリオを並行実行
    scenarios = {
        "Hub": hub_idx,
        "Random": random_idx
    }

    for scenario_name, target_idx in scenarios.items():
        # 初期化: 一様分布 U(0, 0.2) で初期値依存性を排除
        np.random.seed(42) # 比較のためシード固定
        X = np.random.uniform(0.0, 0.2, N)
        D = np.random.uniform(0.0, 0.1, N)
        b_val = np.zeros(N)

        X_history = []

        for step in range(TOTAL_STEPS):
            # フェーズごとのペイオフ制御
            if step == BURN_IN_STEPS:
                b_val[target_idx] = B_PAYLOAD # 攻撃開始
            elif step == BURN_IN_STEPS + ATTACK_STEPS:
                b_val[target_idx] = 0.0       # 攻撃停止 (ヒステリシスの確認)

            X, D = sde_update(X, D, degrees, b_val, N, dt=DT)

            # --- Phase 0: Burn-in (自己組織化のKS統計量記録) ---
            if step < BURN_IN_STEPS and scenario_name == "Hub": # 片方のシナリオで記録すれば十分
                if step % 10 == 0:
                    X_history.append(X.copy())
                    if len(X_history) > 1:
                        stat, _ = ks_2samp(X_history[-1], X_history[-2])
                        ks_results.append({"N": N, "Step": step, "KS_Statistic": stat})

            # 欠陥率(Topological Defect)の記録
            if step % 5 == 0:
                mean_D = np.mean(D)
                defect_results.append({
                    "N": N,
                    "Scenario": scenario_name,
                    "Step": step,
                    "Defect_D": mean_D
                })

# ==========================================
# 4. データ保存と可視化 (CSV & グラフ)
# ==========================================
df_ks = pd.DataFrame(ks_results)
df_defect = pd.DataFrame(defect_results)

df_ks.to_csv(f"{OUTPUT_DIR}/ks_convergence.csv", index=False)
df_defect.to_csv(f"{OUTPUT_DIR}/topological_hysteresis.csv", index=False)

# グラフ1: KS統計量の収束 (キャリブレーション不要の証明)
plt.figure(figsize=(10, 6))
for N in N_LIST:
    subset = df_ks[df_ks["N"] == N]
    plt.plot(subset["Step"], subset["KS_Statistic"], marker='o', label=f"N={N}")
plt.title("Ergodic Self-Organization: KS Statistic Convergence (Burn-in)")
plt.xlabel("Time Step")
plt.ylabel("KS Statistic ($D_{KS}$)")
plt.legend()
plt.grid(True)
plt.savefig(f"{OUTPUT_DIR}/Fig1_KS_Convergence.png", dpi=300)
plt.close()

# グラフ2: ハブ vs ランダムのトポロジー崩壊とヒステリシス (最大スケール)
plt.figure(figsize=(10, 6))
N_max = N_LIST[-1]
subset_hub = df_defect[(df_defect["N"] == N_max) & (df_defect["Scenario"] == "Hub")]
subset_rnd = df_defect[(df_defect["N"] == N_max) & (df_defect["Scenario"] == "Random")]

plt.plot(subset_hub["Step"], subset_hub["Defect_D"], color='red', linewidth=2, label="Hub Origin (Super-spreader)")
plt.plot(subset_rnd["Step"], subset_rnd["Defect_D"], color='blue', linestyle='--', label="Random Origin (Baseline)")

# フェーズ境界線の描画
plt.axvline(x=BURN_IN_STEPS, color='gray', linestyle=':', label="Attack Starts ($b_t=0.9$)")
plt.axvline(x=BURN_IN_STEPS + ATTACK_STEPS, color='black', linestyle=':', label="Attack Ends ($b_t=0.0$)")

# 背景色の設定によるフェーズの可視化
plt.axvspan(0, BURN_IN_STEPS, color='lightgray', alpha=0.3, label="Phase 0: Burn-in")
plt.axvspan(BURN_IN_STEPS, BURN_IN_STEPS + ATTACK_STEPS, color='mistyrose', alpha=0.5, label="Phase 1a: Pathogenesis")
plt.axvspan(BURN_IN_STEPS + ATTACK_STEPS, TOTAL_STEPS, color='lavender', alpha=0.5, label="Phase 1b: Hysteresis")

plt.title(f"Topological Phase Transition & Irreversible Hysteresis (N={N_max})")
plt.xlabel("Time Step")
plt.ylabel("Topological Defect ($D$)")
plt.legend(loc="upper left", fontsize=9)
plt.grid(True)
plt.savefig(f"{OUTPUT_DIR}/Fig2_Hysteresis_Hub_vs_Random.png", dpi=300)
plt.close()

# グラフ3: マクロスケールにおける最終的な非対称性（Asymptotic Bias）
plt.figure(figsize=(8, 5))
final_steps = df_defect[df_defect["Step"] == TOTAL_STEPS - 5]
hub_finals = final_steps[final_steps["Scenario"] == "Hub"]["Defect_D"].values
rnd_finals = final_steps[final_steps["Scenario"] == "Random"]["Defect_D"].values

plt.plot(N_LIST, hub_finals, marker='o', color='red', label="Final Defect (Hub)")
plt.plot(N_LIST, rnd_finals, marker='s', color='blue', linestyle='--', label="Final Defect (Random)")
plt.xscale('log')
plt.title("Asymptotic Bias Across Macroscopic Scales")
plt.xlabel("Network Size N (Log Scale)")
plt.ylabel("Final Topological Defect (Hysteresis Plateau)")
plt.legend()
plt.grid(True)
plt.savefig(f"{OUTPUT_DIR}/Fig3_Asymptotic_Bias_Scaling.png", dpi=300)
plt.close()

# ==========================================
# 5. ZIP圧縮と自動ダウンロード
# ==========================================
shutil.make_archive("Simulation_Results_Hypothesis1", 'zip', OUTPUT_DIR)
print("シミュレーション完了。結果をZIP圧縮してダウンロードします...")
files.download("Simulation_Results_Hypothesis1.zip")

--- Simulating N = 1000 ---
--- Simulating N = 10000 ---
--- Simulating N = 100000 ---
シミュレーション完了。結果をZIP圧縮してダウンロードします...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>